# SmartVision AI — Phase 2: Transfer Learning (25-class classification)

Train **VGG16, ResNet50, MobileNetV2, EfficientNetB0** on the 224×224 classification crops from `Smartvision.ipynb`.

**Runtime:** Google Colab **T4 GPU**. Runtime → Change runtime type → T4 GPU.

**Data:** unzip `MyDrive/smartvision_dataset.zip` produced by `Smartvision.ipynb` (200 quality crops per class, letterbox, 20% box padding).

**Training (submitted EfficientNetB0: 80.7% test accuracy):**
1. Stage 1: freeze the ImageNet backbone, train the head.
2. Stage 2: unfreeze the last blocks only (BatchNorm stays frozen except VGG), Adam `1e-5`.
3. EfficientNetB0 always starts from ImageNet. Stage 2 opens the last **80** layers.

Augmentation is the brief set: horizontal flip, ±15° rotation, brightness, contrast, zoom, color jitter.

**Output:** `*.keras`, `reports/classification_metrics.json`, confusion matrices.

Run order: install → setup → tf.data → helpers → VGG → ResNet → MobileNet → EfficientNet → persist.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install tensorflow pandas scikit-learn matplotlib seaborn pillow tqdm

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, optimizers, callbacks

print("TF", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    if zip_path.exists() and not (data_dir / "classification" / "train").exists():
        import zipfile
        print("Unzipping dataset (reuse Drive zip, not a new collect)...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_dir if not any(data_dir.glob("*")) else PROJECT_ROOT)
        if not (data_dir / "classification").exists():
            inner = PROJECT_ROOT / "smartvision_dataset"
            print("classification path exists:", (inner / "classification").exists())
    DRIVE_REPO = Path("/content/drive/MyDrive/Smart_Vision_AI")
    if DRIVE_REPO.exists():
        sys.path.insert(0, str(DRIVE_REPO))
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = PROJECT_ROOT

CLASS_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
    "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake", "chair",
    "couch", "potted plant", "bed",
]
NUM_CLASSES = 25
IMAGE_SIZE = 224
BATCH = 16  # 16 is safer on Colab T4 with VGG16 flatten; raise to 32 if memory allows

DATA = PROJECT_ROOT / "smartvision_dataset" / "classification"
for alt in (
    Path("/content/smartvision_dataset/classification"),
    Path("/content/drive/MyDrive/smartvision_dataset/classification"),
    Path("/content/drive/MyDrive/SmartVision_artifacts/smartvision_dataset/classification"),
):
    if (DATA / "train").exists():
        break
    if (alt / "train").exists():
        DATA = alt
        break
print("DATA =", DATA, "exists", DATA.exists())

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

print("OUT_ROOT =", OUT_ROOT)
print("Existing checkpoints (resume stage 2 from these if FOUND):")
for stem in ("vgg16", "resnet50", "mobilenetv2", "efficientnetb0"):
    p = MODELS_DIR / f"{stem}.keras"
    if p.exists():
        print(f"  {p.name}: FOUND {p.stat().st_size/1e6:.1f} MB")
    else:
        print(f"  {p.name}: missing — will train stage 1 from ImageNet")

def count_split(split):
    rows = {}
    root = DATA / split
    for n in CLASS_NAMES:
        folder = root / n
        rows[n] = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    return rows

for split in ("train", "val", "test"):
    c = count_split(split)
    print(f"{split:5s} total={sum(c.values()):4d}  min_class={min(c.values())} max_class={max(c.values())} missing={[k for k,v in c.items() if v==0]}")
assert sum(count_split("train").values()) > 0, "No training images. Unzip MyDrive/smartvision_dataset.zip — do not rebuild notebook 1."


In [ ]:
## tf.data pipelines + the brief's augmentation set
# flip, rotation ±15°, brightness ±20%, contrast, zoom, color jitter (saturation/hue)

def list_image_label(split):
    paths, labels = [], []
    root = DATA / split
    for idx, name in enumerate(CLASS_NAMES):
        for f in sorted((root / name).glob("*.jpg")):
            paths.append(str(f))
            labels.append(idx)
    return tf.constant(paths), tf.constant(labels, dtype=tf.int32)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = tf.cast(img, tf.float32)  # 0..255 — model-specific preprocess_input handles scaling
    return img, label

# Keras preprocessing layers: rotation of 15/360 ≈ 0.0417 of a full turn
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15.0 / 360.0, fill_mode="nearest"),
    layers.RandomBrightness(0.20),
    layers.RandomContrast(0.20),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.05, 0.05),
], name="brief_augmentation")

def color_jitter(img, label):
    img = tf.image.random_saturation(img / 255.0, 0.7, 1.3) * 255.0
    img = tf.image.random_hue(img / 255.0, 0.05) * 255.0
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_ds(split, training=False, batch=BATCH):
    paths, labels = list_image_label(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(color_jitter, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)

train_ds, n_train = make_ds("train", training=True)
val_ds, n_val = make_ds("val", training=False)
test_ds, n_test = make_ds("test", training=False)
print(f"n_train={n_train} n_val={n_val} n_test={n_test}")

# Peek one augmented batch so we know augmentation is actually firing
xb, yb = next(iter(train_ds))
print("batch", xb.shape, xb.dtype, "range", float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(tf.cast(tf.clip_by_value(xb[i], 0, 255), tf.uint8))
    ax.set_title(CLASS_NAMES[int(yb[i])], fontsize=8)
    ax.axis("off")
fig.suptitle("Augmented training crops (brief transforms)")
fig.tight_layout()
fig.savefig(FIGURES / "aug_preview.png", dpi=130)
plt.show()

In [ ]:
def clf_loss():
    return "sparse_categorical_crossentropy"


def common_callbacks(name, patience=8, csv_name=None, min_val_acc=None):
    csv = csv_name or name
    ckpt_kw = dict(
        filepath=str(MODELS_DIR / f"{name}.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    )
    try:
        if min_val_acc is not None:
            ckpt = callbacks.ModelCheckpoint(initial_value_threshold=float(min_val_acc), **ckpt_kw)
        else:
            ckpt = callbacks.ModelCheckpoint(**ckpt_kw)
    except TypeError:
        ckpt = callbacks.ModelCheckpoint(**ckpt_kw)
    return [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True, mode="max"),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ckpt,
        callbacks.CSVLogger(str(REPORTS / f"{csv}_history.csv")),
    ]


def plot_history(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.history["accuracy"], label="train")
    ax[0].plot(hist.history["val_accuracy"], label="val")
    ax[0].set_title(f"{name} accuracy"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(hist.history["loss"], label="train")
    ax[1].plot(hist.history["val_loss"], label="val")
    ax[1].set_title(f"{name} loss"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}_history.png", dpi=130)
    plt.show()


def find_backbone(model, substr):
    key = substr.lower()
    for layer in model.layers:
        if key in layer.name.lower():
            return layer
    raise RuntimeError(f"No backbone matching {substr!r} in {[l.name for l in model.layers]}")


def unfreeze_last(backbone, n_unfreeze, freeze_bn=True):
    backbone.trainable = True
    keep = max(0, len(backbone.layers) - n_unfreeze)
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= keep
        if freeze_bn and isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    n_tr = sum(1 for l in backbone.layers if l.trainable)
    print(f"backbone {backbone.name}: trainable {n_tr}/{len(backbone.layers)} (last {n_unfreeze} opened, BN frozen={freeze_bn})")


def compile_clf(model, lr):
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss=clf_loss(),
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")],
    )


def backup_run1(stem):
    src = MODELS_DIR / f"{stem}.keras"
    dst = MODELS_DIR / f"{stem}_run1.keras"
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print("Backed up run-1 weights ->", dst)


def val_accuracy(model):
    out = model.evaluate(val_ds, verbose=0)
    # [loss, accuracy, top5] or [loss, accuracy]
    acc = float(out[1])
    print(f"current val_accuracy={acc:.4f}")
    return acc


def load_or_none(stem):
    path = MODELS_DIR / f"{stem}.keras"
    if path.exists():
        print("Loading Drive checkpoint", path)
        keras.mixed_precision.set_global_policy("float32")
        return keras.models.load_model(path, compile=False)
    return None


def stage2_finetune(model, stem, backbone_substr, n_unfreeze, epochs, lr, freeze_bn=True):
    backup_run1(stem)
    bb = find_backbone(model, backbone_substr)
    unfreeze_last(bb, n_unfreeze, freeze_bn=freeze_bn)
    compile_clf(model, lr)
    floor = val_accuracy(model)
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=common_callbacks(stem, patience=8, csv_name=f"{stem}_stage2", min_val_acc=floor),
        verbose=1,
    )
    plot_history(hist, f"{stem}_stage2")
    return keras.models.load_model(MODELS_DIR / f"{stem}.keras", compile=False)


def evaluate_model(model, name):
    y_true, y_prob = [], []
    t0 = time.perf_counter()
    n = 0
    for xb, yb in test_ds:
        p = model.predict(xb, verbose=0)
        y_true.append(yb.numpy())
        y_prob.append(p)
        n += xb.shape[0]
    elapsed = time.perf_counter() - t0
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = y_prob.argmax(axis=1)
    top5 = np.mean([yt in np.argsort(pr)[-5:] for yt, pr in zip(y_true, y_prob)])
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    prec_c, rec_c, f1_c, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(11, 10))
    sns.heatmap(cm, cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{name} confusion matrix (test)")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIGURES / f"cm_{name}.png", dpi=140)
    plt.show()
    size_mb = (MODELS_DIR / f"{name}.keras").stat().st_size / 1e6 if (MODELS_DIR / f"{name}.keras").exists() else 0.0
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
    print(report)
    per_class = {
        CLASS_NAMES[i]: {"precision": float(prec_c[i]), "recall": float(rec_c[i]), "f1": float(f1_c[i]), "support": int(sup[i])}
        for i in range(NUM_CLASSES)
    }
    metrics = {
        "model": name,
        "accuracy": acc,
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "f1_macro": float(f1),
        "top5_accuracy": float(top5),
        "inference_ms": float(elapsed / max(n, 1) * 1000),
        "model_size_mb": float(size_mb),
        "n_test": int(n),
        "per_class": per_class,
    }
    print(name, {k: metrics[k] for k in ("accuracy", "f1_macro", "top5_accuracy", "inference_ms", "model_size_mb")})
    return metrics, cm

ALL_METRICS = {}
print("Helpers ready.")


In [ ]:
## Model 1 — VGG16: resume Drive checkpoint if present, else frozen head, then unfreeze last 8 (block5)

keras.mixed_precision.set_global_policy("float32")
vgg = load_or_none("vgg16")
if vgg is None:
    print("No checkpoint — stage 1: frozen VGG16 conv base + dropout head")
    base = applications.VGG16(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    base.trainable = False
    inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = applications.vgg16.preprocess_input(inp)
    x = base(x, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    vgg = models.Model(inp, out, name="VGG16")
    compile_clf(vgg, 1e-3)
    hist_vgg1 = vgg.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("vgg16", patience=5), verbose=1)
    plot_history(hist_vgg1, "vgg16")
    vgg = keras.models.load_model(MODELS_DIR / "vgg16.keras")
else:
    print("Skipping stage 1 — using run-1 VGG16 weights from Drive")

print("Stage 2: unfreeze last 8 VGG layers, lr=1e-5")
vgg = stage2_finetune(vgg, "vgg16", "vgg16", n_unfreeze=8, epochs=25, lr=1e-5, freeze_bn=False)
ALL_METRICS["VGG16"], _ = evaluate_model(vgg, "vgg16")


In [ ]:
## Model 2 — ResNet50: resume Drive checkpoint if present, else last-20 warmup, then unfreeze last 40

keras.mixed_precision.set_global_policy("float32")
resnet = load_or_none("resnet50")
if resnet is None:
    print("No checkpoint — stage 1: unfreeze last 20 (brief default) + GAP head")
    base = applications.ResNet50(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    for layer in base.layers:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True
    inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = applications.resnet50.preprocess_input(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    resnet = models.Model(inp, out, name="ResNet50")
    compile_clf(resnet, 1e-4)
    hist_rn1 = resnet.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("resnet50", patience=5), verbose=1)
    plot_history(hist_rn1, "resnet50")
    resnet = keras.models.load_model(MODELS_DIR / "resnet50.keras")
else:
    print("Skipping stage 1 — using run-1 ResNet50 weights from Drive (was 76.5% test)")

print("Stage 2: unfreeze last 40 ResNet layers, lr=1e-5, BN frozen")
resnet = stage2_finetune(resnet, "resnet50", "resnet50", n_unfreeze=40, epochs=30, lr=1e-5, freeze_bn=True)
ALL_METRICS["ResNet50"], _ = evaluate_model(resnet, "resnet50")


In [ ]:
## Model 3 — MobileNetV2: resume Drive checkpoint if present, else frozen head, then unfreeze last 40

keras.mixed_precision.set_global_policy("float32")
mnet = load_or_none("mobilenetv2")
if mnet is None:
    print("No checkpoint — stage 1: frozen MobileNetV2 + compact head")
    base = applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    base.trainable = False
    inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = applications.mobilenet_v2.preprocess_input(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    mnet = models.Model(inp, out, name="MobileNetV2")
    compile_clf(mnet, 1e-3)
    hist_mn1 = mnet.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("mobilenetv2", patience=5), verbose=1)
    plot_history(hist_mn1, "mobilenetv2")
    mnet = keras.models.load_model(MODELS_DIR / "mobilenetv2.keras")
else:
    print("Skipping stage 1 — using run-1 MobileNetV2 weights from Drive")

print("Stage 2: unfreeze last 40 MobileNet layers, lr=1e-5, BN frozen")
mnet = stage2_finetune(mnet, "mobilenetv2", "mobilenet", n_unfreeze=40, epochs=25, lr=1e-5, freeze_bn=True)
ALL_METRICS["MobileNetV2"], _ = evaluate_model(mnet, "mobilenetv2")


In [ ]:
## Model 4 — EfficientNetB0: ImageNet → frozen head → last 80 layers (submitted 80.7%)

keras.mixed_precision.set_global_policy("float32")
print("EfficientNetB0 from ImageNet (do not resume an older checkpoint)")
base = applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
base.trainable = False
inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = applications.efficientnet.preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
eff = models.Model(inp, out, name="EfficientNetB0")
compile_clf(eff, 1e-3)
hist_eff1 = eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=common_callbacks("efficientnetb0", patience=4),
    verbose=1,
)
plot_history(hist_eff1, "efficientnetb0")
eff = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras", compile=False)

print("Stage 2: unfreeze last 80 EfficientNet layers, lr=1e-5, BN frozen")
eff = stage2_finetune(eff, "efficientnetb0", "efficientnet", n_unfreeze=80, epochs=25, lr=1e-5, freeze_bn=True)
ALL_METRICS["EfficientNetB0"], _ = evaluate_model(eff, "efficientnetb0")


In [ ]:
print("EfficientNetB0 two-stage training is the submitted 80% path.")
print("Do not unfreeze the full backbone on this dataset — that overfits.")
for name, m in ALL_METRICS.items():
    print(f"  {name}: test acc={m['accuracy']:.3f}")


In [ ]:
## Persist classification metrics (overwrites reports/classification_metrics.json on Drive)

payload = {
    "class_names": CLASS_NAMES,
    "n_train": int(n_train),
    "n_val": int(n_val),
    "n_test": int(n_test),
    "classification": ALL_METRICS,
    "best_classification_model": max(ALL_METRICS, key=lambda k: ALL_METRICS[k]["accuracy"]),
}
(REPORTS / "classification_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "per_class"} for k, v in ALL_METRICS.items()}, indent=2))
print("Best:", payload["best_classification_model"])
print("Saved models:", sorted(p.name for p in MODELS_DIR.glob("*.keras")))
print("Paste the printed accuracy table back into chat when this cell finishes.")
